# Solutions, Lab 03: Your First Real Run, Classical KD and the Capacity Gap

This notebook solves the four exercises at the end of
`labs/lab-03-first-real-run-classical-kd.ipynb`. Lab 03 is a Tier 2 lab, which means its
training runs execute only on a training box with `RUN_TRAINING = True`. The solutions follow
the same split:

- **Exercise 1 (temperature sweep):** the mechanism is demonstrated live on synthetic logits;
  the four training runs are written in full but gated.
- **Exercise 2 (label smoothing):** the mechanism is demonstrated live on a synthetic
  distribution; the teacher fine-tune and the re-run of the `soft` arm are gated.
- **Exercise 3 (length stratification):** fully live. The measurement runs on the real eval
  set with the released 360M and 135M models in fp32.
- **Exercise 4 (patient teacher):** the config discipline is checked live; the doubled-length
  run is gated, with expected ranges from the capacity-gap literature the lab cites.

That is two exercises with substantial live evidence, one fully live, and one gated. Attempt
the exercises yourself before reading further; the point of a Tier 2 exercise is the judgment
you form while doing it, and a solutions file read first replaces that judgment with mine.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import (hinton_kd_loss, kl_divergence, shift_for_next_token,
                     completion_mask_from_prompt_lens, top1_agreement,
                     mean_entropy, expected_calibration_error, masked_mean)
from kd_pipeline import (set_seed_everywhere, config_fingerprint, MemoryPlan,
                         full_ft_gb, infer_gb, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box, same flag as the lab
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# The lab's configs, copied verbatim so gated solutions run the same experiment.
BASE = dict(
    dataset="HuggingFaceTB/smol-smoltalk", n_train=4096, n_eval=256,
    seq_len=384, lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500,
    warmup_steps=50, T=2.0, dtype="bfloat16",
)
ARMS = {
    "hard":      {**BASE, "teacher": None,                                  "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 0.0},
    "mixed":     {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 0.5},
    "soft":      {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-360M-Instruct", "alpha": 1.0},
    "gap-small": {**BASE, "teacher": "HuggingFaceTB/SmolLM2-360M-Instruct", "student": "HuggingFaceTB/SmolLM2-135M-Instruct", "alpha": 0.5},
    "gap-large": {**BASE, "teacher": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "student": "HuggingFaceTB/SmolLM2-135M-Instruct", "alpha": 0.5},
}

def diff_keys(a: dict, b: dict) -> set:
    return {k for k in a.keys() | b.keys() if a.get(k) != b.get(k)}

print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Exercise 1: Sweep T in {1, 2, 4, 8} on the `soft` arm

**The exercise, restated.** Run the pure-distillation arm four times, changing only the
temperature T (the divisor applied to both models' logits before the softmax; higher T
flattens both distributions). Hinton's paper says intermediate temperatures win. Verify that,
and connect what happens at T=8 to Lab 01 section 3, which showed that as T grows the KD
objective turns into a mean-squared error on the logits themselves.

**The approach.** The sweep itself is a training job, so it is gated. But the *reason*
intermediate T should win is not a training result; it is a property of the loss, and it can
be shown on synthetic logits where I control the teacher exactly. The plan has three
measurements, one per end of the temperature range plus one for the bookkeeping in between:

1. **T=1 is nearly a hard label when the teacher is confident.** I build a teacher that puts
   logit 9 on the correct token, logit 5 on one similar wrong token, and small noise on the
   rest. At T=1 that teacher's distribution puts almost all its mass on the correct token, so
   the wrong-answer structure (the whole point of soft targets) carries almost no probability
   and therefore almost no gradient. Raising T moves mass onto the wrong answers, which is
   what makes their relative probabilities teachable.
2. **The T squared factor keeps the comparison fair.** The gradient of the softened KL shrinks
   roughly like 1 over T squared (each softmax at T divides its input by T, and there are two
   of them in the chain rule). I measure the raw gradient norm with and without the
   compensation to show the shrinkage and its repair. Without the repair, a T sweep would also
   be a learning-rate sweep, and the comparison would be meaningless.
3. **High T is logit matching, not distribution matching.** Lab 01 section 3's identity: as T
   grows, the gradient of the compensated soft loss approaches the gradient of a mean-squared
   error between mean-centered logits. I verify this numerically by computing both gradients
   at T=64 and checking their cosine similarity (the cosine of the angle between the two
   gradient vectors: 1.0 means identical direction). Why this predicts T=8 losing: logit MSE
   weights every vocabulary entry equally, so the student spends capacity matching the
   teacher's arbitrary logit values on tokens the teacher itself considers irrelevant, which
   is noise for a next-token objective.

If all three facts hold, the gated sweep's expected shape (poor at T=1, best at T=2 or 4,
declining by T=8) follows from the loss itself, and the training run only confirms it.

In [2]:
set_seed_everywhere(SEED)

# A confident teacher with structure in the wrong answers: correct token at logit 9,
# one "similar" wrong token at logit 5, everything else small noise.
V, B, P = 32, 2, 6
correct = torch.randint(0, V, (B, P))
similar = (correct + 1) % V
zt = 0.5 * torch.randn(B, P, V)
zt.scatter_(-1, correct.unsqueeze(-1), torch.full((B, P, 1), 9.0))
zt.scatter_(-1, similar.unsqueeze(-1), torch.full((B, P, 1), 5.0))
zs0 = torch.randn(B, P, V)          # an untrained student, fixed across all T
m = torch.ones(B, P, dtype=torch.bool)

def soft_grad(T, scaled):
    z = zs0.clone().requires_grad_(True)
    kl_divergence(z, zt, m, T=T, scale_by_T2=scaled).backward()
    return z.grad

print(f"{'T':>3} {'p(correct)':>11} {'wrong mass':>11} {'entropy':>8} "
      f"{'|grad| raw':>11} {'|grad| xT^2':>11}")
stats = {}
for T in (1, 2, 4, 8):
    p = F.softmax(zt / T, -1)
    pc = float(p.gather(-1, correct.unsqueeze(-1)).mean())
    H = float(-(p * p.clamp_min(1e-12).log()).sum(-1).mean())
    g_raw = float(soft_grad(T, False).norm())
    g_sc = float(soft_grad(T, True).norm())
    stats[T] = dict(pc=pc, wrong=1 - pc, g_raw=g_raw, g_sc=g_sc)
    print(f"{T:>3} {pc:>11.4f} {1-pc:>11.4f} {H:>8.3f} {g_raw:>11.5f} {g_sc:>11.5f}")

# 1: at T=1 this confident teacher IS nearly a hard label.
assert stats[1]["pc"] > 0.95, "T=1 soft target should be nearly one-hot here"
# and raising T moves teachable mass onto the wrong answers.
assert stats[8]["wrong"] > 5 * stats[1]["wrong"]

# 2: without the T^2 factor the gradient dies like 1/T^2; with it, it stays comparable.
raw_ratio = stats[1]["g_raw"] / stats[8]["g_raw"]
sc_ratio = stats[1]["g_sc"] / stats[8]["g_sc"]
assert 20 < raw_ratio < 300, f"raw gradient should shrink ~T^2=64x, got {raw_ratio:.0f}x"
assert 0.3 < sc_ratio < 4.0, f"compensated gradient should stay same-scale, got {sc_ratio:.2f}x"

# 3: the high-T limit is logit MSE (Lab 01 section 3). Compare gradient directions.
def centered_mse_grad():
    z = zs0.clone().requires_grad_(True)
    zc = z - z.mean(-1, keepdim=True)
    tc = zt - zt.mean(-1, keepdim=True)
    masked_mean(((zc - tc) ** 2).mean(-1), m).backward()
    return z.grad

cos = F.cosine_similarity
g_mse = centered_mse_grad().flatten()
cos64 = float(cos(soft_grad(64.0, True).flatten(), g_mse, dim=0))
cos1 = float(cos(soft_grad(1.0, True).flatten(), g_mse, dim=0))
assert cos64 > 0.99, f"T=64 KD gradient should align with logit-MSE gradient, cos={cos64:.4f}"
assert cos64 > cos1, "the alignment must be a high-T effect, not always true"
print(f"\nraw-gradient shrinkage T=1 vs T=8: {raw_ratio:.0f}x (T^2 would be 64x); "
      f"compensated: {sc_ratio:.2f}x")
print(f"cos(KD grad, logit-MSE grad): {cos64:.4f} at T=64 vs {cos1:.4f} at T=1")
print("CHECK ex1-live: T=1 hides dark knowledge, T^2 keeps scales fair, "
      "high T becomes logit MSE")

  T  p(correct)  wrong mass  entropy  |grad| raw |grad| xT^2
  1      0.9779      0.0221    0.130     0.28616     0.28616
  2      0.6748      0.3252    1.612     0.09652     0.38606
  4      0.2189      0.7811    3.162     0.01485     0.23760
  8      0.0878      0.9122    3.420     0.00244     0.15597

raw-gradient shrinkage T=1 vs T=8: 117x (T^2 would be 64x); compensated: 1.83x
cos(KD grad, logit-MSE grad): 0.9996 at T=64 vs 0.8118 at T=1
CHECK ex1-live: T=1 hides dark knowledge, T^2 keeps scales fair, high T becomes logit MSE


In [3]:
# Exercise 1, gated part: the actual sweep on the soft arm, T in {1, 2, 4, 8}.
# Full working code, identical to the lab's Part B except that only T moves.
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup

def kd_step(student, teacher, batch, alpha, T):
    # One knowledge-distillation step, copied from the lab's Part B.
    ids, labels, m_ = batch["input_ids"], batch["labels"], batch["mask"]
    s_logits = student(ids).logits
    if teacher is not None:
        with torch.no_grad():
            t_logits = teacher(ids).logits
    else:
        t_logits = s_logits.detach()
    s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, m_)
    return hinton_kd_loss(s_sh, t_sh, labels[:, 1:], m_sh, T=T, alpha=alpha)

@torch.no_grad()
def evaluate(student, teacher, eval_batches):
    aggs = {"agree": [], "fwd_kl": [], "entropy": [], "ece": []}
    for batch in eval_batches:
        s_logits = student(batch["input_ids"]).logits
        t_logits = teacher(batch["input_ids"]).logits if teacher is not None else s_logits
        s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, batch["mask"])
        aggs["agree"].append(top1_agreement(s_sh, t_sh, m_sh))
        aggs["fwd_kl"].append(float(kl_divergence(s_sh, t_sh, m_sh, scale_by_T2=False)))
        aggs["entropy"].append(mean_entropy(s_sh, m_sh))
        aggs["ece"].append(expected_calibration_error(s_sh, batch["input_ids"][:, 1:], m_sh))
    return {k: sum(v) / len(v) for k, v in aggs.items()}

def run_arm(name, cfg, out_root="../runs/sol03"):
    set_seed_everywhere(SEED)
    dtype = getattr(torch, cfg["dtype"])
    student = AutoModelForCausalLM.from_pretrained(cfg["student"], dtype=dtype).to(device)
    teacher = None
    if cfg["teacher"]:
        teacher = AutoModelForCausalLM.from_pretrained(cfg["teacher"], dtype=dtype)
        teacher = teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    tr = torch.load("../data/lab03/train.pt"); ev = torch.load("../data/lab03/eval.pt")
    to_batches = lambda d, bs: [
        {k: d[k][i:i+bs].to(device) for k in ("input_ids", "labels", "mask")}
        for i in range(0, len(d["input_ids"]), bs)]
    train_batches = to_batches(tr, cfg["batch_size"])
    eval_batches = to_batches(ev, cfg["batch_size"])
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    log, step = [], 0
    while step < cfg["max_steps"]:
        for batch in train_batches:
            loss = kd_step(student, teacher, batch, cfg["alpha"], cfg["T"]) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 100 == 0:
                metrics = evaluate(student, teacher, eval_batches[:4])
                log.append({"step": step, "loss": float(loss) * cfg["grad_accum"], **metrics})
                print(f"[{name}] step {step:>5}  loss {log[-1]['loss']:.3f}  "
                      f"agree {metrics['agree']:.3f}  ECE {metrics['ece']:.3f}")
            step += 1
            if step >= cfg["max_steps"]:
                break
    fp = config_fingerprint({**cfg, "seed": SEED})
    out = os.path.join(out_root, f"{name}_{fp}")
    os.makedirs(out, exist_ok=True)
    student.save_pretrained(out)
    json.dump(log, open(os.path.join(out, "log.json"), "w"), indent=2)
    RunManifest(name=name, config=cfg, seed=SEED,
                artifacts_in={"train": "lab03/train.pt"},
                artifacts_out={"checkpoint": out}).save(out)
    del student, teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    return out

T_ARMS = {f"soft-T{T}": {**ARMS["soft"], "T": float(T)} for T in (1, 2, 4, 8)}
names = list(T_ARMS)
for i, a in enumerate(names):
    for b in names[i+1:]:
        assert diff_keys(T_ARMS[a], T_ARMS[b]) == {"T"}, "the sweep must move T only"
print("sweep discipline verified: 4 arms differing in T only")

if RUN_TRAINING:
    sweep_out = {name: run_arm(name, cfg) for name, cfg in T_ARMS.items()}
    print(json.dumps(sweep_out, indent=2))
else:
    print("RUN_TRAINING=False: the T sweep is written but did not execute here.")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sweep discipline verified: 4 arms differing in T only
RUN_TRAINING=False: the T sweep is written but did not execute here.


**Interpretation.** The live table is the whole argument, one row per temperature. At T=1
the synthetic teacher put over 95 percent of its probability on the correct token (the printed
`p(correct)` column), which means the soft target was carrying almost no wrong-answer
information; distilling it is close to training on hard labels, so the T=1 arm should track
the `hard` arm from the lab. By T=8 more than 90 percent of the mass sat on wrong answers and
the entropy approached ln 32, which is about 3.47 nats, the entropy of a uniform distribution
over the 32-token vocabulary; the target is dissolving toward uniform. The gradient columns
showed the raw soft gradient shrinking by roughly two orders of magnitude between T=1 and T=8
(the T squared prediction is exactly 64x) while the compensated gradient stayed within a small
factor, which is why the gated sweep is a fair comparison at all. And the cosine similarity
above 0.99 at T=64, against 0.82 at T=1, confirms Lab 01 section 3's identity: high-T KD is
logit MSE, an objective that treats the teacher's logit on its 30,000th-choice token as
exactly as important as its logit on the answer.

**The gated sweep did not execute in this build; the following are expected ranges, not
results.** Grounding them in the lab's Part C: the soft arm at T=2 beat hard labels by 2 to 8
points of top-1 agreement, so expect `soft-T2` and `soft-T4` in that band, usually within a
point or two of each other, with either one on top being a legitimate result. `soft-T1`
should land closest to the `hard` baseline of all four, for the reason the first table row
shows: its targets are nearly hard labels. `soft-T8` should give back some of the T=2 gain
(roughly 1 to 4 points) and often shows the worst ECE of the four, because logit-style
matching unties confidence from correctness. Failure signature to watch: if T=8 wins your
sweep outright, check your teacher's entropy first; a teacher that is unusually diffuse to
begin with leaves less room for flattening to hurt, and that is a property of the teacher,
not a refutation of Hinton.

## Exercise 2: Give the teacher label smoothing

**The exercise, restated.** Fine-tune the 1.7B teacher briefly with label smoothing 0.1, then
re-run the `soft` arm from this smoothed teacher. Müller et al.'s result predicts the student
gets *worse* even though the teacher itself is unchanged or better on accuracy. Explain the
effect through what smoothing does to the wrong-answer probabilities.

**The approach.** Label smoothing needs a definition before anything else: instead of
training the teacher toward a target of 1.0 on the correct token and 0 elsewhere, smoothing
with epsilon = 0.1 trains it toward 0.9 on the correct token and 0.1 spread *uniformly* over
the whole vocabulary. Read that target carefully: every wrong token gets exactly the same
share, epsilon over V. A teacher optimized against that target is being explicitly rewarded
for making all wrong answers equally probable. But the entire value of a soft target, from
the lab's opening section, lives in the wrong answers being *unequally* probable: "cat" a
thousand times more likely than "car" when the answer is "dog" is the similarity structure
the student absorbs. Smoothing trains that structure away on purpose.

The fine-tune is a gated training job, but the mechanism is pure arithmetic on
distributions, so the live cell does this: build a teacher distribution with realistic
structure (a confident correct token, one plausible wrong answer, a long tail), then compute
what the smoothing-optimal teacher would emit at the same position, and a halfway point to
stand in for a brief fine-tune that only partly reaches the optimum. For each, measure the
two numbers the mechanism turns on: the probability ratio between a plausible wrong answer
and an implausible one, and the standard deviation of the log-probabilities across all wrong
answers (a one-number summary of how much wrong-answer structure exists; zero means all
wrong answers are interchangeable).

In [4]:
set_seed_everywhere(SEED)

# One position of a realistic teacher: answer "dog" at index 0, plausible wrong
# answer "cat" at index 1, implausible "car" at index 2, and a structured tail.
V = 1000
zt = 2.0 * torch.randn(V)
zt[0], zt[1], zt[2] = 9.0, 4.0, -3.0
p = F.softmax(zt, 0)

eps = 0.1
p_smooth_opt = torch.full((V,), eps / V)      # the smoothing-optimal teacher output
p_smooth_opt[0] += 1 - eps
p_half = 0.5 * p + 0.5 * p_smooth_opt         # a brief fine-tune: partway there

def wrong_answer_stats(dist, name):
    ratio = float(dist[1] / dist[2])                  # p(cat) / p(car)
    spread = float(dist[1:].log().std())              # structure among ALL wrong answers
    print(f"{name:<28} p(dog)={float(dist[0]):.3f}  p(cat)/p(car)={ratio:>9.1f}  "
          f"std of wrong log-probs={spread:.3f}")
    return ratio, spread

r0, s0 = wrong_answer_stats(p, "original teacher")
r1, s1 = wrong_answer_stats(p_half, "briefly smoothed (halfway)")
r2, s2 = wrong_answer_stats(p_smooth_opt, "smoothing optimum")

# The correct answer survives smoothing untouched: same argmax, similar confidence.
assert int(p.argmax()) == int(p_smooth_opt.argmax()) == 0, "smoothing does not change top-1"
# The wrong-answer structure does not survive: the ratio collapses toward 1
# and the spread collapses toward 0.
assert r0 > 100, "the original teacher has strong similarity structure"
assert r2 < 1.001, "at the smoothing optimum every wrong answer is interchangeable"
assert s1 < 0.6 * s0 and s2 < 1e-6, "smoothing destroys wrong-answer structure monotonically"
print("\nCHECK ex2-live: smoothing preserves the teacher's answer and erases its "
      "dark knowledge (ratio -> 1, spread -> 0)")

original teacher             p(dog)=0.556  p(cat)/p(car)=   1096.6  std of wrong log-probs=2.019
briefly smoothed (halfway)   p(dog)=0.728  p(cat)/p(car)=     37.2  std of wrong log-probs=0.963
smoothing optimum            p(dog)=0.900  p(cat)/p(car)=      1.0  std of wrong log-probs=0.000

CHECK ex2-live: smoothing preserves the teacher's answer and erases its dark knowledge (ratio -> 1, spread -> 0)


In [5]:
# Exercise 2, gated part: fine-tune the teacher with label smoothing, then
# re-run the soft arm from the smoothed teacher. Full working code.
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup

def finetune_teacher_smoothed(cfg, smoothing=0.1, steps=300, out_dir="../runs/sol03/teacher_ls"):
    set_seed_everywhere(SEED)
    teacher = AutoModelForCausalLM.from_pretrained(
        cfg["teacher"], dtype=getattr(torch, cfg["dtype"])).to(device)
    tr = torch.load("../data/lab03/train.pt")
    opt = torch.optim.AdamW(teacher.parameters(), lr=1e-5)
    sched = get_cosine_schedule_with_warmup(opt, 20, steps)
    bs, step = cfg["batch_size"], 0
    while step < steps:
        for i in range(0, len(tr["input_ids"]), bs):
            ids = tr["input_ids"][i:i+bs].to(device)
            labels = tr["labels"][i:i+bs].to(device)
            logits = teacher(ids).logits[:, :-1]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   labels[:, 1:].reshape(-1), ignore_index=-100,
                                   label_smoothing=smoothing)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(teacher.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1
            if step % 50 == 0:
                print(f"[teacher-ls] step {step:>4}  smoothed CE {float(loss):.3f}")
            if step >= steps:
                break
    teacher.save_pretrained(out_dir)
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    return out_dir

if RUN_TRAINING:
    ls_dir = finetune_teacher_smoothed(ARMS["soft"])
    soft_ls = {**ARMS["soft"], "teacher": ls_dir}
    assert diff_keys(soft_ls, ARMS["soft"]) == {"teacher"}, "one variable: the teacher"
    out = run_arm("soft-from-smoothed-teacher", soft_ls)
    print("smoothed-teacher arm ->", out)
else:
    print("RUN_TRAINING=False: teacher fine-tune and re-distillation are written "
          "but did not execute here.")

RUN_TRAINING=False: teacher fine-tune and re-distillation are written but did not execute here.


**Interpretation.** The live table shows the mechanism in three rows. The original
teacher held a wrong-answer ratio p(cat)/p(car) near 1100 (the printed check), meaning it
ranked the plausible mistake three orders of magnitude above the implausible one, and its
wrong-answer log-probabilities had a standard deviation around 2 nats of structure. The
smoothing optimum keeps p(dog) high and the argmax identical, so any accuracy-style
evaluation of the teacher sees nothing wrong, while the ratio collapses to exactly 1.0 and
the spread to exactly 0: every wrong answer becomes the same wrong answer. Even the halfway
point, standing in for a brief fine-tune, cut the ratio by a factor of about 30 and the
spread by half. That is Müller et al.'s finding reduced to arithmetic: label smoothing
improves the teacher's own calibration by *discarding* exactly the information distillation
transfers. A teacher can get better and become a worse teacher at the same time, because
"teacher quality" for KD is measured in the wrong answers.

**The gated fine-tune and re-run did not execute in this build; expected ranges, not
results:** relative to the lab's `soft` arm, the smoothed-teacher student should lose 1 to 4
points of top-1 agreement (against the *original* 1.7B as the reference, so the comparison is
apples to apples) and its advantage over the `hard` baseline should shrink or vanish, while
the smoothed teacher's own eval cross-entropy stays flat or improves slightly. Confirming
evidence is that ordering; refuting evidence would be the smoothed-teacher student matching
the original within noise, which with a 300-step fine-tune is possible simply because the
teacher did not move far, so before concluding against Müller, verify the teacher actually
smoothed: recompute the live cell's spread statistic on the fine-tuned teacher's real logits
over a few eval rows and confirm it dropped.

## Exercise 3: Length-stratify the eval

**The exercise, restated.** Split the eval rows by completion length (the number of
supervised assistant tokens in the row) and recompute top-1 agreement per stratum. The claim
to test: KD's advantage typically concentrates on longer completions. Why?

**The approach.** This build machine has no trained Lab 03 checkpoints (`../runs/lab03` was
never populated here), so I split the exercise into the part that is fully measurable now and
the part that needs the checkpoints. The measurable part is the entire measurement machinery:
stratify the real eval set by completion length, then compute per-stratum top-1 agreement
between a real model pair in fp32, using the released SmolLM2-360M-Instruct as the reference
(teacher role) and SmolLM2-135M-Instruct as the model under test (student role). These are
the `gap-small` arm's pair before any training, so the numbers below are that arm's *starting
line*: the agreement a KD run would improve upon, stratified the same way you would stratify
the trained checkpoint. The gated cell afterward applies the identical function to trained
checkpoints once they exist.

Mechanically: completion length is `mask.sum()` per row. I sort the 256 eval rows by that
length, cut them into thirds (short, medium, long), and take 20 rows spread evenly through
each third so each stratum spans its range rather than clustering at a boundary. Agreement is
computed per stratum with the same shift-then-mask discipline as the lab: logits at position
t are scored against the mask entry for token t+1. Everything runs in fp32 because this
machine has no GPU and fp32 on CPU is the numerically boring choice.

In [6]:
import gc
from transformers import AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

set_seed_everywhere(SEED)
ev = torch.load("../data/lab03/eval.pt")
comp_len = ev["mask"].sum(1)                       # supervised tokens per row
order = comp_len.argsort()
n = len(order)
thirds = [order[:n//3], order[n//3:2*n//3], order[2*n//3:]]
PER, BS = 20, 4
pick = lambda t: t[torch.linspace(0, len(t) - 1, PER).long()]
strata = {name: pick(t) for name, t in zip(("short", "medium", "long"), thirds)}

@torch.no_grad()
def argmaxes(model_name, rows):
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32).eval()
    outs = []
    for i in range(0, len(rows), BS):
        ids = ev["input_ids"][rows[i:i+BS]]
        outs.append(model(ids).logits[:, :-1].argmax(-1))   # shifted: predicts token t+1
    del model
    gc.collect()
    return torch.cat(outs)

all_rows = torch.cat([strata[s] for s in strata])
t_arg = argmaxes("HuggingFaceTB/SmolLM2-360M-Instruct", all_rows)
s_arg = argmaxes("HuggingFaceTB/SmolLM2-135M-Instruct", all_rows)

print(f"{'stratum':>8} {'len range':>12} {'rows':>5} {'positions':>10} {'agreement':>10}")
results, off = {}, 0
for name in ("short", "medium", "long"):
    rows = strata[name]
    m_sh = ev["mask"][rows][:, 1:]                 # mask shifted with the logits
    sl = slice(off, off + len(rows)); off += len(rows)
    agree = float(((t_arg[sl] == s_arg[sl]) & m_sh).sum() / m_sh.sum())
    lens = comp_len[rows]
    n_pos = int(m_sh.sum())
    results[name] = agree
    print(f"{name:>8} {int(lens.min()):>5}-{int(lens.max()):>6} {len(rows):>5} "
          f"{n_pos:>10} {agree:>10.3f}")
    assert len(rows) == PER and n_pos > 200, "each stratum must be well populated"
    assert 0.2 < agree < 0.98, "agreement should be informative, not degenerate"

spread = max(results.values()) - min(results.values())
print(f"\nCHECK ex3-live: per-stratum agreement measured on real models; "
      f"spread across strata = {spread:.3f} "
      f"({'length matters' if spread > 0.02 else 'flat in length'} for this untrained pair)")

 stratum    len range  rows  positions  agreement
   short     9-    77    20        902      0.823
  medium    78-   139    20       2114      0.821
    long   139-   332    20       3854      0.849

CHECK ex3-live: per-stratum agreement measured on real models; spread across strata = 0.029 (length matters for this untrained pair)


In [7]:
# Exercise 3, gated part: the same stratified measurement on trained checkpoints.
# Runs once Lab 03's Part B has produced ../runs/lab03/<arm>_<fingerprint>/ directories.
from transformers import AutoModelForCausalLM as _AM

def stratified_agreement(student_dir, teacher_name, strata_dict):
    t_a = argmaxes(teacher_name, torch.cat(list(strata_dict.values())))
    s_a = argmaxes(student_dir, torch.cat(list(strata_dict.values())))
    out, off = {}, 0
    for name, rows in strata_dict.items():
        m_sh = ev["mask"][rows][:, 1:]
        sl = slice(off, off + len(rows)); off += len(rows)
        out[name] = float(((t_a[sl] == s_a[sl]) & m_sh).sum() / m_sh.sum())
    return out

if RUN_TRAINING and os.path.isdir("../runs/lab03"):
    ckpts = sorted(os.listdir("../runs/lab03"))
    for ck in ckpts:
        arm = ck.split("_")[0]
        teacher = ARMS.get(arm, {}).get("teacher") or ARMS["soft"]["teacher"]
        table = stratified_agreement(os.path.join("../runs/lab03", ck), teacher, strata)
        print(ck, {k: round(v, 3) for k, v in table.items()})
else:
    print("Gated: rerun with RUN_TRAINING=True after Lab 03 Part B has produced "
          "checkpoints in ../runs/lab03; this cell then prints the same table per arm.")

Gated: rerun with RUN_TRAINING=True after Lab 03 Part B has produced checkpoints in ../runs/lab03; this cell then prints the same table per arm.


**Interpretation.** The live table gives the untrained baseline this pair starts from,
and it already shows the shape the exercise asks about: the short and medium strata landed
within a fraction of a point of each other while the long stratum ran about 3 points higher
(the printed spread line is the check; on this build it came out just under 0.03 between the
extremes). Two things are
worth separating in that number. First, per-position difficulty is not uniform along a
completion: the first few tokens after a prompt are the highest-entropy positions, because
many continuations are still plausible, while deep inside a long completion the local context
(the sentence being finished, the list being continued, the code block being closed) pins the
next token down for models of any size. Short completions are made almost entirely of the
hard early positions; long completions dilute them with easy late positions. So a per-token
agreement average mechanically favors long strata. That is the honest caveat on the
measurement machinery, and it applies to trained checkpoints exactly as it applies here.

Second, the exercise's actual claim is about the KD *advantage*, the gap between a distilled
arm and the `hard` arm, stratum by stratum, and that is what the gated cell computes once
checkpoints exist. **That comparison did not execute in this build; expected shape, not
results:** the distilled-minus-hard gap should be larger on the long stratum than the short
one, typically by a factor of 1.5 to 3 on this model family. The reason follows from what
each loss teaches. Hard labels supervise one token per position and say nothing about the
alternatives, so on the pinned-down late positions of long completions the hard-trained
student already agrees and there is nothing to gain; the soft target's extra information is
the teacher's *ranking of plausible alternatives*, and that ranking is exactly what
mid-completion positions with several defensible continuations need. On top of that,
per-token averaging weights long rows more heavily during training too (the lab's
`masked_mean` note), so the distilled student simply spent more of its gradient budget there.
If your trained table instead shows the KD advantage concentrating on *short* completions,
the usual culprit is prompt leakage into the mask; re-run the lab's four-point audit before
believing it.

## Exercise 4: The patient teacher

**The exercise, restated.** Double `max_steps` on the `gap-large` arm only (1.7B teacher into
a 135M student, the 12.6x capacity gap). Beyer et al.'s "patient teacher" result predicts the
capacity gap partially closes when the student is given more optimization time. Does it?

**The approach.** This is purely a training question; there is no synthetic stand-in that
honestly demonstrates a capacity-gap closure, so the run is gated and the live part of the
solution is the experimental discipline: the patient arm must differ from `gap-large` in
exactly one key (`max_steps`), and its comparison targets must be pinned before running. The
comparisons that matter are three, and it is worth being precise because the naive one is
wrong. Comparing `gap-large-patient` at step 3000 with `gap-large` at step 1500 confounds
patience with compute. The clean readings are: (a) patient at step 1500 versus `gap-large` at
step 1500, which must match (same config to that point, same seed) and is a free
reproducibility check; (b) patient at 3000 versus `gap-large` at 1500, the exercise's
question, does more time help; and (c) patient at 3000 versus `gap-small` at 1500, the
capacity-gap question, does patience buy back what the oversized teacher cost.

In [8]:
patient = {**ARMS["gap-large"], "max_steps": ARMS["gap-large"]["max_steps"] * 2}
assert diff_keys(patient, ARMS["gap-large"]) == {"max_steps"}, "one variable: patience"
assert patient["max_steps"] == 3000 and patient["warmup_steps"] == 50
print(f"gap-large        fingerprint {config_fingerprint({**ARMS['gap-large'], 'seed': SEED})}")
print(f"gap-large-patient fingerprint {config_fingerprint({**patient, 'seed': SEED})}")
print("CHECK ex4-live: patient arm isolates max_steps as the only moving variable")

if RUN_TRAINING:
    out = run_arm("gap-large-patient", patient)
    print("patient arm ->", out)
else:
    print("RUN_TRAINING=False: the doubled-length run is written but did not execute here.")

gap-large        fingerprint 7ca101f1cc63
gap-large-patient fingerprint 2e06180f6a2f
CHECK ex4-live: patient arm isolates max_steps as the only moving variable
RUN_TRAINING=False: the doubled-length run is written but did not execute here.


**Interpretation.** The live check is small but load-bearing: with `max_steps` as the
only differing key and the same seed, the patient run's first 1500 steps are a re-execution
of `gap-large`, so any drift there is a reproducibility bug, not a finding.

**The run did not execute in this build; expected ranges, not results,** grounded in the
lab's Part C and the literature it cites. Beyer et al.'s result (distillation is a
"consistent and patient teacher" problem; their gains came from training far longer than
supervised recipes suggest) predicts the second 1500 steps still help: expect
`gap-large-patient` at 3000 steps to gain roughly 1 to 3 points of top-1 agreement over
`gap-large` at 1500, with the eval KL still declining slowly rather than flat. The
capacity-gap comparison is the interesting one: Part C expected `gap-large` to beat
`gap-small` by less than the teacher-size ratio suggests, or to lose outright; patience
should close part of whatever deficit appeared, and *partial* closure is the published
pattern, so patient-gap-large landing between `gap-large` and `gap-small` confirms the
exercise's premise. Full closure or overtaking would say the gap in your setup was mostly an
optimization-time artifact, which is a real and reportable result. The failure signature to
watch in the second half of the run is the cosine learning-rate schedule: doubling
`max_steps` stretches the schedule, so the patient run spends its extra steps at moderate
learning rates rather than the near-zero tail; if you instead resume a finished run and bolt
on 1500 extra steps with a fresh schedule, the comparison to (b) is no longer clean. And if
agreement gains but ECE worsens through the extension, the student is memorizing the
teacher's confidence faster than its ranking, which is the known late-phase failure of long
distillation runs; stop at the ECE inflection, not at the step budget.